# Chapter 11: Causal Language Modeling

[Read this chapter online](https://jackluu.io/book/section-3-the-transformer/ch11-causal-language-modeling/) &nbsp;|&nbsp; [Open in Colab](https://colab.research.google.com/github/jackluucoding/build-llm-from-zero/blob/main/notebooks/ch11-causal-language-modeling.ipynb)

From *Building an LLM from Zero: Look Inside the Black Box* by Truong (Jack) Luu.


In [ ]:
# Run me first. Safe to run more than once; it skips whatever is already done.
import os
import subprocess
import sys

FOLDER = "build-llm-from-zero"

# 1. Fetch the code, unless we are already inside it
if os.path.basename(os.getcwd()) != FOLDER:
    if not os.path.isdir(FOLDER):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackluucoding/build-llm-from-zero"], check=True)
    os.chdir(FOLDER)
sys.path.insert(0, os.getcwd())

# 2. PyTorch, the CPU build, which is all this book needs
try:
    import torch
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)

# 3. The Shakespeare text
subprocess.run([sys.executable, "src/utils/download_data.py"], check=True)

print("Ready. Working in", os.getcwd())

# Chapter 11: Causal Language Modeling

![You are here: Next-Token Scores](../assets/diagrams/ch11-where-we-are.png){ width="756" }
*Figure 11.1: The model is built. Now we define its goal: predicting the next token.*

Now that we have assembled the full Transformer architecture in Chapter 10, our engine is built. But an engine is useless without a task (Figure 11.1). In this chapter, we give our model its objective: causal language modeling. This simply means guessing the next token based only on what came before it. We will also learn how to measure its performance.

In this chapter you will:

- Shift the training sequence to create input and target pairs.
- Understand cross-entropy loss as a measure of surprise.
- See how generation is just a loop of predicting and appending.

**Words to Know**
    - **Logits**: The raw, unnormalized scores the model outputs before they are turned into probabilities.
    - **Softmax**: A mathematical function that squashes a list of any numbers into positive percentages that add up to 100%.
    - **Cross-Entropy Loss**: A mathematical way to measure how wrong the model's predictions are. Lower is better.
    - **Autoregressive**: A process that uses its own past outputs as inputs for its next step.

## Theory

### The Training Trick: One Pass, Many Examples

How do we train a model to predict the next token? You might think we feed it one token, ask for the next, check the answer, and then feed it two tokens. That would be incredibly slow.

The brilliant insight of causal language modeling is that a single pass over a sequence of length `T` gives us `T` training examples all at once.

![Sequence shifted to create targets](../assets/diagrams/ch11-shifted-targets.png){ width="737" }
*Figure 11.2: One sequence provides multiple training examples simultaneously.*

As shown in Figure 11.2, the model processes the whole sequence. Because of the causal mask (from Chapter 6), position 2 cannot see position 3. Each position only sees what came before it. Therefore, at every single position, the model can make a valid prediction about what comes next.

### Input and Target: The Shifted Pair

To implement this efficiently, we take our sequence of text and create two slightly different copies:

1.  **Input (`x`)**: The sequence missing its very last token.
2.  **Target (`y`)**: The sequence missing its very first token.

This means that at any position `i`, the character in the input $x[i]$ is supposed to predict the character in the target $y[i]$. If our sequence is "HELLO", `x` is "HELL" and `y` is "ELLO". At position 0, "H" predicts "E". At position 1, "E" (with "H" as context) predicts "L", and so on.

### Logits and Probabilities

When our model makes predictions, it does not immediately output a single character or a clean percentage. It outputs raw numbers called logits.

![Logits turned into probabilities and loss](../assets/diagrams/ch11-logits-to-probabilities.png){ width="738" }
*Figure 11.3: Softmax converts raw scores into valid percentages.*

Logits can be any number: negative, positive, small, or large. To make sense of them, as illustrated in Figure 11.3, we pass them through a function called **softmax**. Softmax squashes all the logits so they are positive and sum to exactly 1.0 (or 100%). Now we have a probability distribution over our 65 possible characters.

### Measuring Success: Cross-Entropy Loss

Once we have probabilities, we need to know how well the model is doing. We use a metric called **cross-entropy loss**.

Think of loss as a measure of the model's surprise. If the correct next character is "A", and the model assigned a 99% probability to "A", it is not surprised at all. The loss is very low. If it assigned a 1% probability to "A", it is highly surprised. The loss is very high.

If a model is completely untrained and guessing blindly among our 65 characters, it will assign roughly equal probability (about 1.5%) to each. The mathematical loss for this complete ignorance is about 4.17. We want our training process to push this number down.

This brings us to the end of the "Next-Token Scores" stage on our map. Our model now makes predictions and measures its own mistakes, setting the stage for it to learn.

## Code

![Code flow: logits and targets into cross-entropy](../assets/diagrams/ch11-code-flow.png){ width="618" }
*Figure 11.4: The model produces logits for the input, which are compared to the target to calculate loss.*

Let's look at how this is implemented (following the flow in Figure 11.4).

```python
x = example_ids[:-1]
    y = example_ids[1:]

    logits = model(token_ids)

    # Calculate loss by comparing predictions to actual targets
    loss = F.cross_entropy(
        logits.view(B * T_len, config.vocab_size),
        targets.view(B * T_len)
    )
```

And how to run the full script:

```python
$ python src/ch10_causal_lm.py
--- 1. Constructing input/target pairs ---
Sequence: [20, 17, 30, 30, 33, 1, 35, 53, 56, 30]
Input  x: [20, 17, 30, 30, 33, 1, 35, 53, 56]
Target y: [17, 30, 30, 33, 1, 35, 53, 56, 30]
At each position i, x[i] predicts y[i].

--- 2. Computing cross-entropy loss ---
Logits shape : torch.Size([4, 20, 65])
Targets shape: torch.Size([4, 20])
Loss (random model): 4.3070
Expected loss for random: 4.1744

--- 3. Text generation ---
We extend a starting sequence one token at a time.

Generated IDs (first 10): [0, 54, 34, 5, 55, 18, 63, 52, 43, 10]
Output shape: torch.Size([1, 51])
After training (Chapter 13), this will produce real text!
```

**What just happened:**

1.  Lines 1 and 2 slice a short sequence of 10 tokens into an input `x` of length 9 and a target `y` of length 9 to demonstrate input/target pairs.
2.  Line 4 passes a random batch of 20 tokens into the untrained model to get logits for the loss computation step.
3.  Lines 7 to 10 calculate the cross-entropy loss, which is 4.3070, very close to our expected random guessing loss of 4.1744.
4.  The output shows we generated 50 tokens. The machinery works; the weights are still random, so the characters are too.

**Shape Check:**

(Note: The shapes below are from the loss computation step, which uses 20 tokens, unlike the 10-token sequence used earlier. Table 11.1 lists the shapes at each step.)

**Table 11.1:** Tensor shapes for the cross-entropy loss calculation.

| Variable | Shape | Meaning |
| :--- | :--- | :--- |
| `token_ids` | `[4, 20]` | 4 batches of 20 tokens each. |
| `logits` | `[4, 20, 65]` | A score for each of the 65 possible next characters, at every position. |
| `loss` | `[]` | A single scalar number representing the average surprise. |

## Try It

How does confidence affect the loss? We can simulate different prediction scenarios manually.

```python
"""Calculate cross-entropy loss for different confidence levels."""
import torch
import torch.nn.functional as F

# Two possible words: 'A' (index 0) or 'B' (index 1)
target = torch.tensor([0]) # The correct answer is 'A'

print("Scenario 1: Guessing blindly (50% / 50%)")
logits_blind = torch.tensor([[0.0, 0.0]])
loss_blind = F.cross_entropy(logits_blind, target)
print(f"Loss: {loss_blind.item():.4f}")

print("\nScenario 2: Confident and right (88% for 'A')")
logits_right = torch.tensor([[2.0, 0.0]])
loss_right = F.cross_entropy(logits_right, target)
print(f"Loss: {loss_right.item():.4f}")

print("\nScenario 3: Confident and wrong (88% for 'B')")
logits_wrong = torch.tensor([[0.0, 2.0]])
loss_wrong = F.cross_entropy(logits_wrong, target)
print(f"Loss: {loss_wrong.item():.4f}")
```

Lines 14 and 15 calculate the loss when the model is confident and correct, resulting in a much lower loss.

```python
$ python src/examples/ch11_loss_demo.py
Scenario 1: Guessing blindly (50% / 50%)
Loss: 0.6931

Scenario 2: Confident and right (88% for 'A')
Loss: 0.1269

Scenario 3: Confident and wrong (88% for 'B')
Loss: 2.1269
```

**Try It**
    Open `src/examples/ch11_loss_demo.py`. Change `logits_right` to `[[5.0, 0.0]]` to make the model even more confident. Run the script and see how close to zero the loss gets.

**In Business**
    Imagine you are building an assistant to draft text in your company's house style. Causal language modeling is how the assistant learns to write like you. By reviewing thousands of past emails and reports (the target sequences), it learns which words typically follow other words in your organization's specific context. The loss tells you how close its drafts are to your actual historical data.

**Watch Out**
    When calculating cross-entropy loss in PyTorch, the `F.cross_entropy` function expects the logits to be flattened. It wants a 2D tensor of shape `[Total Tokens, Vocabulary Size]`, not a 3D tensor of `[Batch, Time, Vocabulary Size]`. That is why the code uses `logits.view(B * T_len, config.vocab_size)`. Forgetting to reshape is a very common bug!

## Key Takeaways

- A single forward pass on a sequence of length `T` provides `T` separate training examples, making training highly efficient.
- The target sequence is simply the input sequence shifted one position into the future.
- The model outputs raw logits, which softmax converts into probabilities.
- Cross-entropy loss measures how "surprised" the model is by the correct answer. We want this number to be as low as possible.
- An untrained model guessing among 65 characters will have a loss of approximately 4.17.

## Check Your Understanding

1. If your sequence is "DATA", what is the input sequence `x` and the target sequence `y`?
2. Why do we need softmax before we can interpret the model's output as percentages?
3. If a model is perfectly confident and perfectly correct, what should its cross-entropy loss be?


## Further Reading

**Train once, reuse everywhere.** Labeled examples are expensive, and every new task used to need its own pile of them. BERT trained one Transformer on ordinary text by hiding words and asking it to fill the blanks, then adapted that single model to many tasks with a small amount of labeled data each. It reads in both directions at once, which is exactly what the causal mask in Chapter 11 forbids: masking is what separates a model that fills blanks from one that writes forward.

<div class="refs" markdown>

Devlin, J., Chang, M.-W., Lee, K., & Toutanova, K. (2018). *BERT: Pre-training of deep bidirectional transformers for language understanding* (arXiv:1810.04805). arXiv. https://doi.org/10.48550/arXiv.1810.04805

</div>

---

### `src/ch10_causal_lm.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/ch10_causal_lm.py"   # a cell has none, and the file uses it to find the text

"""
Train the model to predict the next token and generate text.
This file belongs to Chapter 11.
Run: python src/ch10_causal_lm.py
"""
import torch
import torch.nn.functional as F
import os
import sys

from src.utils.config import GPTConfig
from src.ch09_gpt_model import GPT

# Settings
config = GPTConfig()

# --- The Idea ---
def generate(model, start_ids, max_new_tokens):
    # Set model to evaluation mode (e.g. disable dropout)
    model.eval()
    context = start_ids.clone()

    for _ in range(max_new_tokens):
        # Crop context to the maximum block size the model can handle
        ctx = context[:, -config.block_size:]
        logits = model(ctx)

        # Focus on the very last time step to predict the next token
        next_logits = logits[:, -1, :]
        probs = F.softmax(next_logits, dim=-1)

        # Randomly sample the next token based on probabilities
        next_id = torch.multinomial(probs, num_samples=1)

        # Append the new token to the sequence
        context = torch.cat([context, next_id], dim=1)

    return context

# --- Demo ---
if __name__ == "__main__":
    torch.manual_seed(42)
    print("Chapter 11: Causal Language Modeling and Generation\n")

    print("--- 1. Constructing input/target pairs ---")
    example_ids = torch.tensor([20, 17, 30, 30, 33, 1, 35, 53, 56, 30])
    T = len(example_ids)

    x = example_ids[:-1]
    y = example_ids[1:]

    print(f"Sequence: {example_ids.tolist()}")
    print(f"Input  x: {x.tolist()}")
    print(f"Target y: {y.tolist()}")
    print("At each position i, x[i] predicts y[i].")

    print("\n--- 2. Computing cross-entropy loss ---")
    model = GPT(config)

    B, T_len = 4, 20
    token_ids = torch.randint(0, config.vocab_size, (B, T_len))
    targets   = torch.randint(0, config.vocab_size, (B, T_len))

    logits = model(token_ids)

    # Calculate loss by comparing predictions to actual targets
    loss = F.cross_entropy(
        logits.view(B * T_len, config.vocab_size),
        targets.view(B * T_len)
    )

    print(f"Logits shape : {logits.shape}")
    print(f"Targets shape: {targets.shape}")
    print(f"Loss (random model): {loss.item():.4f}")
    print(f"Expected loss for random: "
          f"{torch.log(torch.tensor(config.vocab_size)):.4f}")

    print("\n--- 3. Text generation ---")
    print("We extend a starting sequence one token at a time.")

    start = torch.zeros((1, 1), dtype=torch.long)
    output_ids = generate(model, start, max_new_tokens=50)

    print(f"\nGenerated IDs (first 10): {output_ids[0, :10].tolist()}")
    print(f"Output shape: {output_ids.shape}")
    print("After training (Chapter 13), this will produce real text!")
    print("\nCausal LM done! Ready for Chapter 12.")

---

### `src/examples/ch11_loss_demo.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/examples/ch11_loss_demo.py"   # a cell has none, and the file uses it to find the text

"""Calculate cross-entropy loss for different confidence levels."""
import torch
import torch.nn.functional as F

# Two possible words: 'A' (index 0) or 'B' (index 1)
target = torch.tensor([0]) # The correct answer is 'A'

print("Scenario 1: Guessing blindly (50% / 50%)")
logits_blind = torch.tensor([[0.0, 0.0]])
loss_blind = F.cross_entropy(logits_blind, target)
print(f"Loss: {loss_blind.item():.4f}")

print("\nScenario 2: Confident and right (88% for 'A')")
logits_right = torch.tensor([[2.0, 0.0]])
loss_right = F.cross_entropy(logits_right, target)
print(f"Loss: {loss_right.item():.4f}")

print("\nScenario 3: Confident and wrong (88% for 'B')")
logits_wrong = torch.tensor([[0.0, 2.0]])
loss_wrong = F.cross_entropy(logits_wrong, target)
print(f"Loss: {loss_wrong.item():.4f}")